# Homework 5

# Задача №1 - Можете ли вы отличить сорняки от рассады?

Теперь приступим к задаче классификации на картинках. Реализуйте программу, которая определяет тип рассады на изображении. 

Для того, чтобы определить характерные особенности каждого типа рассады, у вас есть train. Train это папка, в которой картинки уже классифицированы и лежат в соответствующих папках. Исходя из этой информации можете найти признаки, присущие конкретному растению.

Проверка вашего решения будет на происходить на test. В папке test уже нет метки класса для каждой картинки. 

[Ссылка на Яндекс-диск](https://yadi.sk/d/0Zzp0klXT0iRmA), все картинки тут.

Примеры изображений для теста:
<table><tr>
    <td> <img src="https://i.ibb.co/tbqR37m/fhj.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/6yL3Wmt/sfg.png" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="https://i.ibb.co/pvn7NvF/asd.png" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [6]:
import cv2
import os
import numpy as np
from collections import defaultdict
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

def extract_sift_features(img_path, use_mask=True):
    """Извлекает SIFT-дескрипторы с опциональной маской зелёного цвета"""
    img = cv2.imread(img_path)
    if img is None:
        return None
    
    if use_mask:
        # Создаём маску для зелёных областей (растений)
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, (35, 50, 50), (85, 255, 255))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        gray = cv2.bitwise_and(gray, gray, mask=mask)
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    sift = cv2.SIFT_create()
    _, descriptors = sift.detectAndCompute(gray, None)
    return descriptors

def train_kmeans(train_folder, n_clusters=100, random_state=42):
    """Обучает K-Means на SIFT-дескрипторах из train"""
    all_descriptors = []
    
    for class_name in os.listdir(train_folder):
        class_path = os.path.join(train_folder, class_name)
        if not os.path.isdir(class_path):
            continue
            
        for img_name in os.listdir(class_path):
            img_path = os.path.join(class_path, img_name)
            descriptors = extract_sift_features(img_path)
            if descriptors is not None:
                all_descriptors.append(descriptors)    
    
    all_descriptors = np.vstack(all_descriptors)
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(all_descriptors)
    return kmeans

def image_to_histogram(img_path, kmeans, n_clusters):
    """Преобразует изображение в гистограмму визуальных слов"""
    descriptors = extract_sift_features(img_path)
    
    visual_words = kmeans.predict(descriptors)
    hist, _ = np.histogram(visual_words, bins=n_clusters, range=(0, n_clusters))
    return hist / hist.sum() if hist.sum() > 0 else hist


In [7]:
train_folder = "plants/train"
test_folder = "plants/test"
output_folder = "results"
n_clusters = 100
random_state = 42

print("Обучение K-Means...")
kmeans = train_kmeans(train_folder, n_clusters, random_state)

print("Подготовка данных...")
X_train, y_train = [], []
class_names = sorted(os.listdir(train_folder))

for label, class_name in enumerate(class_names):
    class_path = os.path.join(train_folder, class_name)
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        hist = image_to_histogram(img_path, kmeans, n_clusters)
        X_train.append(hist)
        y_train.append(label)    

print("Обучение классификатора...")
classifier = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=random_state))
])
classifier.fit(X_train, y_train)

print("Классификация тестовых изображений...")
results = defaultdict(list)

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)
    hist = image_to_histogram(img_path, kmeans, n_clusters)
    proba = classifier.predict_proba([hist])[0]
    pred_class = np.argmax(proba)
    results[class_names[pred_class]].append(img_name)    

print("Сохранение результатов...")
os.makedirs(output_folder, exist_ok=True)

for class_name, images in results.items():
    class_dir = os.path.join(output_folder, class_name)
    os.makedirs(class_dir, exist_ok=True)
    
    for img_name in images:
        src_path = os.path.join(test_folder, img_name)
        dst_path = os.path.join(class_dir, img_name)
        img = cv2.imread(src_path)
        if img is not None:
            cv2.imwrite(dst_path, img)

print("Результаты сохранены в папку 'results'.")

Обучение K-Means...
Подготовка данных...
Обучение классификатора...
Классификация тестовых изображений...
Сохранение результатов...
Результаты сохранены в папку 'results'.


# Задача №2 - Собери пазл (2.0).

Даны кусочки изображения, ваша задача склеить пазл в исходную картинку. 

Условия:
* Дано исходное изображение для проверки, использовать собранное изображение в самом алгоритме нельзя;
* Картинки имеют друг с другом пересечение;
* После разрезки кусочки пазлов не были повернуты или отражены;
* НЕЛЬЗЯ выбрать опорную картинку для сбора пазла, как это было в homework 3
* В процессе проверки решения пазлы могут быть перемешаны, т.е. порядок пазлов в проверке может отличаться от исходного 

Изображения расположены по [ссылке](https://disk.yandex.ru/d/XtpawH1sV9UDlg).

Примеры изображений:
<img src="puzzle/su_fighter.jpg" alt="Drawing" style="width: 300px;"/>
<table><tr>
    <td> <img src="puzzle/su_fighter_shuffle/0.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/1.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/2.jpg" alt="Drawing" style="width: 200px;"/> </td>
    <td> <img src="puzzle/su_fighter_shuffle/3.jpg" alt="Drawing" style="width: 200px;"/> </td>
</tr></table>

In [8]:
import cv2
import numpy as np
import os
from skimage.metrics import structural_similarity as ssim

def load_pieces(folder):
    """Загружает все фрагменты пазла из указанной папки"""
    pieces = []
    for filename in sorted(os.listdir(folder)):
        img = cv2.imread(os.path.join(folder, filename))
        if img is not None:
            pieces.append(img)
    return pieces

def get_edge(img, direction, overlap=20):
    """Возвращает край изображения в указанном направлении"""
    if direction == 'top':
        return img[:overlap, :]
    elif direction == 'bottom':
        return img[-overlap:, :]
    elif direction == 'left':
        return img[:, :overlap]
    elif direction == 'right':
        return img[:, -overlap:]
    return None

def compare_edges(edge1, edge2):
    """Сравнивает два края с помощью SSIM"""
    if edge1.shape != edge2.shape:
        return -1
    return ssim(edge1, edge2, multichannel=True, win_size=3)

def merge_images(img1, img2, direction, overlap=20):
    """Объединяет два изображения в указанном направлении"""
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    
    if direction == 'right':
        new_img = np.zeros((max(h1, h2), w1 + w2 - overlap, 3), dtype=np.uint8)
        new_img[:h1, :w1] = img1
        new_img[:h2, w1-overlap:] = img2
    elif direction == 'left':
        new_img = np.zeros((max(h1, h2), w1 + w2 - overlap, 3), dtype=np.uint8)
        new_img[:h2, :w2] = img2
        new_img[:h1, w2-overlap:] = img1
    elif direction == 'bottom':
        new_img = np.zeros((h1 + h2 - overlap, max(w1, w2), 3), dtype=np.uint8)
        new_img[:h1, :w1] = img1
        new_img[h1-overlap:, :w2] = img2
    elif direction == 'top':
        new_img = np.zeros((h1 + h2 - overlap, max(w1, w2), 3), dtype=np.uint8)
        new_img[:h2, :w2] = img2
        new_img[h2-overlap:, :w1] = img1
    
    return new_img

def assemble_puzzle(pieces):
    """Собирает пазл из фрагментов"""
    if not pieces:
        return None
    
    # Начинаем с первого фрагмента
    assembled = pieces[0]
    used_indices = {0}
    
    while len(used_indices) < len(pieces):
        found_match = False
        
        for i in range(len(pieces)):
            if i in used_indices:
                continue
                
            # Проверяем все 4 стороны
            for direction in ['top', 'right', 'bottom', 'left']:
                edge = get_edge(assembled, direction)
                piece_edge = get_edge(pieces[i], get_opposite_direction(direction))
                
                score = compare_edges(edge, piece_edge)
                
                if score > 0.5:  # Порог схожести
                    assembled = merge_images(assembled, pieces[i], direction)
                    used_indices.add(i)
                    found_match = True
                    break
                    
            if found_match:
                break
                
        if not found_match:
            print("Не удалось найти подходящий фрагмент для сборки")
            break
            
    return assembled

def get_opposite_direction(direction):
    """Возвращает противоположное направление"""
    opposites = {'top': 'bottom', 'bottom': 'top', 
                'left': 'right', 'right': 'left'}
    return opposites.get(direction, direction)

def save_result(image, output_folder, filename="su_fighter_restored.jpg"):
    """Сохраняет собранное изображение"""
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    cv2.imwrite(os.path.join(output_folder, filename), image)



pieces1 = load_pieces("puzzle/su_fighter_shuffle")
pieces2 = load_pieces("puzzle/china_shuffle")
pieces3 = load_pieces("puzzle/home_shuffle")


result1 = assemble_puzzle(pieces1)
result2 = assemble_puzzle(pieces2)
result3 = assemble_puzzle(pieces3)


if result1 is not None:
    save_result(result1, "restored_photo", "su_fighter_restored.jpg")
    print("Пазл 1 успешно собран")
else:
    print("Не удалось собрать пазл 1")

if result2 is not None:
    save_result(result2, "restored_photo", "china_restored.jpg")
    print("Пазл 2 успешно собран")
else:
    print("Не удалось собрать пазл 2")

if result3 is not None:
    save_result(result3, "restored_photo", "home_restored.jpg")
    print("Пазл 3 успешно собран")
else:
    print("Не удалось собрать пазл 3")

print(f"Размер result1: {result1.shape if result1 is not None else 'None'}")
print(f"Размер result2: {result2.shape if result2 is not None else 'None'}")
print(f"Размер result3: {result3.shape if result3 is not None else 'None'}")

Пазл 1 успешно собран
Не удалось собрать пазл 2
Не удалось собрать пазл 3
Размер result1: (220, 820, 3)
Размер result2: None
Размер result3: None
